# Week 3  
For Week 3, include concepts such as linear regression with forward and backward selection, PCR, and PLSR. 

#### Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.pipeline import Pipeline

from sklearn.metrics import mean_absolute_error

#### Load Data

In [2]:
diabetes_indicators = pd.read_csv('Modified_Datasets/diabetes_indicators_2.csv')

In [3]:
cervical_cancer = pd.read_csv('Modified_Datasets/cervical_cancer_2.csv')

#### Forward Selection

**Diabetes**

In [4]:
# X, y
Xd = diabetes_indicators.drop('Diabetes_012', axis = 1)
yd = diabetes_indicators['Diabetes_012']
Xd_train, Xd_test, yd_train, yd_test = train_test_split(Xd, yd, test_size = 0.2, random_state = 42)

In [5]:
# CV
cv = KFold(n_splits = 2, shuffle = True, random_state = 42)

In [6]:
# Model
model = LinearRegression()

# Forward selection
forward_selection = SequentialFeatureSelector(
    model, 
    n_features_to_select = 5, 
    cv = cv, 
    n_jobs = -1, 
    direction = "forward"
)
# Set output to dataframe
forward_selection.set_output(transform = 'pandas')

# Fit on data
forward_selection.fit(Xd_train, yd_train)

# Get columns that were selected
features_selected = forward_selection.get_feature_names_out()
print(features_selected)

['HighBP' 'HighChol' 'BMI' 'GenHlth' 'Age']


In [7]:
# Transform X_train, X_test
Xd_train_selected = forward_selection.transform(Xd_train)
Xd_test_selected = forward_selection.transform(Xd_test)

In [8]:
# Would train model after this, skipping for the purpose of illustrating forward & backward selection

**Cervical Cancer**

In [9]:
# X, y
Xc = cervical_cancer.drop('Dx:Cancer', axis = 1)
yc = cervical_cancer['Dx:Cancer']
Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size = 0.2, random_state = 42)

In [10]:
# CV
cv = KFold(n_splits = 2, shuffle = True, random_state = 42)

In [11]:
# Model
model = LinearRegression()

# Forward selection
forward_selection = SequentialFeatureSelector(
    model, 
    n_features_to_select = 10, 
    cv = cv, 
    n_jobs = -1, 
    direction = "forward"
)
# Set output to dataframe
forward_selection.set_output(transform = 'pandas')

# Fit on data
forward_selection.fit(Xc_train, yc_train)

# Get columns that were selected
features_selected = forward_selection.get_feature_names_out()
print(features_selected)

['Number of sexual partners' 'Num of pregnancies' 'Smokes (years)' 'IUD'
 'STDs:vaginal condylomatosis' 'STDs:Hepatitis B' 'Dx:CIN' 'Dx:HPV' 'Dx'
 'Citology']


In [12]:
# Transform X_train, X_test
X_train_selected = forward_selection.transform(Xc_train)
X_test_selected = forward_selection.transform(Xc_test)

#### Backward Selection

**Diabetes**

In [13]:
# CV
cv = KFold(n_splits = 2, shuffle = True, random_state = 42)

In [14]:
# Model
model = LinearRegression()

# Backward selection
backward_selection = SequentialFeatureSelector(
    model, 
    n_features_to_select = 10, 
    cv = cv, 
    n_jobs = -1, 
    direction = "backward"
)

# Set output to dataframe
backward_selection.set_output(transform = 'pandas')

# Fit on data
backward_selection.fit(Xd_train, yd_train)

# Get columns that were selected
features_selected = backward_selection.get_feature_names_out()
print(features_selected)

['HighBP' 'HighChol' 'BMI' 'HeartDiseaseorAttack' 'HvyAlcoholConsump'
 'GenHlth' 'DiffWalk' 'Sex' 'Age' 'Income']


In [15]:
# Convert to DataFrame using the exact feature names learned during fit
Xd_train_df = pd.DataFrame(Xd_train, columns=forward_selection.feature_names_in_)
Xd_test_df = pd.DataFrame(Xd_test, columns=forward_selection.feature_names_in_)

In [16]:
# Transform X_train, X_test
X_train_selected = forward_selection.transform(Xd_train_df)
X_test_selected = forward_selection.transform(Xd_test_df)

**Cervical Cancer**

In [17]:
# CV
cv = KFold(n_splits = 2, shuffle = True, random_state = 42)

In [18]:
# Model
model = LinearRegression()

# Backward selection
backward_selection = SequentialFeatureSelector(
    model, 
    n_features_to_select = 10, 
    cv = cv, 
    n_jobs = -1, 
    direction = "backward"
)

# Set output to dataframe
backward_selection.set_output(transform = 'pandas')

# Fit on data
backward_selection.fit(Xc_train, yc_train)

# Get columns that were selected
features_selected = backward_selection.get_feature_names_out()
print(features_selected)

['Age' 'Num of pregnancies' 'Hormonal Contraceptives' 'IUD' 'IUD (years)'
 'STDs:genital herpes' 'STDs:molluscum contagiosum' 'STDs:Hepatitis B'
 'Dx:CIN' 'Dx']


In [19]:
# Transform X_train, X_test
X_train_selected = forward_selection.transform(Xc_train)
X_test_selected = forward_selection.transform(Xc_test)

#### PCR

**Diabetes**

In [20]:
# PCR pipeline
pcr_model = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components = 10)),
    ('regressor', LinearRegression())
])

In [21]:
# Fit
pcr_model.fit(Xd_train, yd_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('pca', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](22,)","['Unnamed: 0','HighBP','HighChol',...,'Age','Education','Income']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,22
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


In [22]:
# Predict
yd_pred = pcr_model.predict(Xd_test)

In [23]:
# Scoring
print(mean_absolute_error(yd_test, yd_pred))

0.47165205671349825


**Cervical Cancer**

In [24]:
# PCR pipeline
pcr_model = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components = 10)),
    ('regressor', LinearRegression())
])

In [25]:
# Fit
pcr_model.fit(Xc_train, yc_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('pca', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](28,)","['Unnamed: 0','Age','Number of sexual partners',...,'Schiller','Citology', 'Biopsy']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,28
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


In [26]:
# Predict
yc_pred = pcr_model.predict(Xc_test)

In [27]:
# Scoring
print(mean_absolute_error(yc_test, yc_pred))

0.035853315358322556


##### PLSR

**Diabetes**

In [28]:
# Initiate model
model = PLSRegression(
    n_components = 10,
    scale = True
)

In [29]:
# Fit
model.fit(Xd_train, yd_train)

,"n_components n_components: int, default=2Number of components to keep. Should be in `[1, n_features]`.",10
,"scale scale: bool, default=TrueWhether to scale `X` and `y`.",True
,"max_iter max_iter: int, default=500The maximum number of iterations of the power method when`algorithm='nipals'`. Ignored otherwise.",500
,"tol tol: float, default=1e-06The tolerance used as convergence criteria in the power method: thealgorithm stops whenever the squared norm of `u_i - u_{i-1}` is lessthan `tol`, where `u` corresponds to the left singular vector.",1e-06
,"copy copy: bool, default=TrueWhether to copy `X` and `y` in :term:`fit` before applying centering,and potentially scaling. If `False`, these operations will be doneinplace, modifying both arrays.",True
Name,Type,Value
"coef_ coef_: ndarray of shape (n_target, n_features)The coefficients of the linear model such that `y` is approximated as`y = X @ coef_.T + intercept_`.","ndarray[float64](1, 22)","[[ 0. , 0.16, 0.13,..., 0.02,-0.01,-0.01]]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](22,)","['Unnamed: 0','HighBP','HighChol',...,'Age','Education','Income']"
"intercept_ intercept_: ndarray of shape (n_targets,)The intercepts of the linear model such that `y` is approximated as`y = X @ coef_.T + intercept_`... versionadded:: 1.1","ndarray[float64](1,)",[0.33]
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`.,int,22
"n_iter_ n_iter_: list of shape (n_components,)Number of iterations of the power method, for eachcomponent.",list,"[1, 1, 1, 1, ...]"


In [30]:
# Predict
yd_pred = model.predict(Xd_test)

In [31]:
# Scoring
print(mean_absolute_error(yd_test, yd_pred))

0.47365649684301575


**Cervical Cancer**

In [32]:
# Initiate model
model = PLSRegression(
    n_components = 10,
    scale = True
)

In [33]:
# Fit
model.fit(Xc_train, yc_train)

,"n_components n_components: int, default=2Number of components to keep. Should be in `[1, n_features]`.",10
,"scale scale: bool, default=TrueWhether to scale `X` and `y`.",True
,"max_iter max_iter: int, default=500The maximum number of iterations of the power method when`algorithm='nipals'`. Ignored otherwise.",500
,"tol tol: float, default=1e-06The tolerance used as convergence criteria in the power method: thealgorithm stops whenever the squared norm of `u_i - u_{i-1}` is lessthan `tol`, where `u` corresponds to the left singular vector.",1e-06
,"copy copy: bool, default=TrueWhether to copy `X` and `y` in :term:`fit` before applying centering,and potentially scaling. If `False`, these operations will be doneinplace, modifying both arrays.",True
Name,Type,Value
"coef_ coef_: ndarray of shape (n_target, n_features)The coefficients of the linear model such that `y` is approximated as`y = X @ coef_.T + intercept_`.","ndarray[float64](1, 28)","[[-0. , 0. ,-0. ,...,-0. ,-0.01, 0.01]]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](28,)","['Unnamed: 0','Age','Number of sexual partners',...,'Schiller','Citology', 'Biopsy']"
"intercept_ intercept_: ndarray of shape (n_targets,)The intercepts of the linear model such that `y` is approximated as`y = X @ coef_.T + intercept_`... versionadded:: 1.1","ndarray[float64](1,)",[0.02]
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`.,int,28
"n_iter_ n_iter_: list of shape (n_components,)Number of iterations of the power method, for eachcomponent.",list,"[1, 1, 1, 1, ...]"


In [34]:
# Predict
yc_pred = model.predict(Xc_test)

In [35]:
# Scoring
print(mean_absolute_error(yc_test, yc_pred))

0.018275524145265447
